In [4]:
%run init_notebook.py
import sys
sys.path.append('src')

import os
import torch
import torch.optim as optim
import torchaudio.transforms as T
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from IPython.display import clear_output
from tqdm import tqdm

from src.dataset import NSynth, nsynth_collate_fn
from src.models2 import ConditionalVAE
from src.losses import MultiDecoderLoss

In [5]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"listos en: {DEVICE}")

EPOCHS = 50
BATCH_SIZE = 16
LR = 1e-4

os.makedirs("checkpoints", exist_ok=True)

# Cargamos el dataset pidiendo que solo traiga los que tienen el .pt extraído
print("Cargando dataset...")
ds = NSynth('training', require_features=True)
loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=nsynth_collate_fn, num_workers=2)

# Transformación al vuelo a Mel-spec
mel_transform = T.MelSpectrogram(
    sample_rate=16000, 
    n_fft=1024, 
    hop_length=160, 
    n_mels=80
).to(DEVICE)

listos en: cuda
Cargando dataset...
Carga completada: 289205 muestras de todos los instrumentos.


In [6]:
# El modelo híbrido
model = ConditionalVAE(channels=[1, 32, 64, 128, 256]).to(DEVICE)

# La loss: beta_max lo ponemos en 0.1 y que tarde un par de epochs en alcanzarlo
loss_fn = MultiDecoderLoss(beta_max=0.1, beta_steps=len(loader)*5).to(DEVICE) 

optimizer = optim.Adam(model.parameters(), lr=LR)

# Listas para guardar el historial y pintarlo luego
history = {'total': [], 'mel': [], 'ddsp': [], 'kld': []}

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    
    # Barra de progreso integrada en el notebook
    pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for batch in pbar:
        # 1. Desempaquetar
        waveforms, srs, keys, metadatas, features, conditions = batch
        
        # 2. Subir a GPU
        waveforms = waveforms.to(DEVICE)
        f0_real = features['f0'].to(DEVICE)
        loudness_real = features['loudness_db'].to(DEVICE)
        
        inst_oh = conditions['instrument_onehot'].to(DEVICE)
        pitch_n = conditions['pitch_norm'].to(DEVICE)
        vel_n = conditions['velocity_norm'].to(DEVICE)
        bright = conditions['brightness'].to(DEVICE)
        sustain = conditions['sustain'].to(DEVICE)

        # 3. Audio a Mel (y lo recortamos a 128 frames)
        mel_orig = mel_transform(waveforms)
        mel_orig = mel_orig[:, :, :, :128]
        
        # Empaquetamos features reales para la loss
        features_real = {'f0': f0_real, 'loudness_db': loudness_real}

        # --- FORWARD ---
        optimizer.zero_grad()
        mel_hat, ddsp_params, kld = model(mel_orig, inst_oh, pitch_n, vel_n, bright, sustain)
        
        # --- CÁLCULO DEL ERROR ---
        loss, loss_dict = loss_fn(mel_orig, mel_hat, ddsp_params, features_real, kld)

        # --- BACKWARD ---
        loss.backward()
        optimizer.step()
        loss_fn.step_beta() # Aumentar el KLD poco a poco

        # --- LOGGING ---
        epoch_loss += loss.item()
        
        # Guardamos en el historial cada 50 steps para que la gráfica tenga buena resolución
        if loss_fn.current_step % 50 == 0:
            history['total'].append(loss_dict['loss'])
            history['mel'].append(loss_dict['mel'])
            history['ddsp'].append(loss_dict['ddsp'])
            history['kld'].append(loss_dict['kld'])

        pbar.set_postfix({
            'L': f"{loss_dict['loss']:.3f}",
            'Mel': f"{loss_dict['mel']:.3f}", 
            'DDSP': f"{loss_dict['ddsp']:.3f}"
        })

    print(f"Epoch {epoch+1} terminada. Loss media: {epoch_loss/len(loader):.4f}")
    
    # Guardamos checkpoint
    if (epoch + 1) % 5 == 0:
        torch.save(model.state_dict(), f"checkpoints/cvae_epoch_{epoch+1}.pt")

Epoch 1/50:   0%|          | 0/18076 [00:00<?, ?it/s]

In [ ]:
import torch

# Mini-función para sacar los números de la GPU si se han quedado ahí atrapados
def clean_history(tensor_list):
    return [x.item() if torch.is_tensor(x) else x for x in tensor_list]

plt.figure(figsize=(12, 6))

plt.plot(clean_history(history['total']), label='Loss Total', color='black', linewidth=2)
plt.plot(clean_history(history['mel']), label='Loss Mel (Espectrograma)', alpha=0.8)
plt.plot(clean_history(history['ddsp']), label='Loss DDSP (f0 + Loudness)', alpha=0.8)
plt.plot(clean_history(history['kld']), label='Loss KLD (Espacio Latente)', alpha=0.8, linestyle='--')

plt.title('Evolución del Entrenamiento CVAE')
plt.xlabel('Steps (x50)')
plt.ylabel('Pérdida')
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.show()

NameError: name 'history' is not defined

<Figure size 1200x600 with 0 Axes>